<H3>PRI 2025/26: first project delivery</H3>

**GROUP X**
- name, number
- name, number
- name, number

<H3>Part 0: data loading</H3>

In [ ]:

import os
from pathlib import Path

def load_bbc_dataset(root_dir):
    root = Path(root_dir)
    docs = []
    refs = []
    for category in ["business", "entertainment", "politics", "sport", "tech"]:
        art_dir = root / "News Articles" / category
        sum_dir = root / "Summaries" / category
        for art_file in sorted(art_dir.glob("*.txt")):
            sum_file = sum_dir / art_file.name
            if not sum_file.exists():
                continue
            # more robust decoding
            with art_file.open(encoding="utf-8", errors="replace") as f:
                d = f.read().strip()
            with sum_file.open(encoding="utf-8", errors="replace") as f:
                r = f.read().strip()
            docs.append({"id": art_file.stem, "category": category, "text": d})
            refs.append({"id": art_file.stem, "category": category, "summary": r})
    return docs, refs
   
## test 
docs, refs = load_bbc_dataset("data/BBC News Summary")
print(len(docs), len(refs))
print(docs[0])
print(refs[0])




2225 2225
{'id': '001', 'category': 'business', 'text': 'Ad sales boost Time Warner profit\n\nQuarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.\n\nThe firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.\n\nTime Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL\'s underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service f

<H3>Part I: demo of facilities</H3>

A) **Indexing** (preprocessing and indexing options)

In [6]:
import time
import sys
import spacy
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    doc = nlp(text)

    tokens = [
        token.lemma_.lower()
        for token in doc
        if token.is_alpha and not token.is_stop
    ]

    noun_phrases = [
        chunk.text.lower().replace(" ", "_")
        for chunk in doc.noun_chunks
    ]

    return tokens + noun_phrases


def indexing(D, args=None):

    start = time.time()

    processed_docs = {}
    inverted_index = defaultdict(list)

    texts_for_tfidf = []
    doc_ids = []

    for doc in D:

        doc_id = doc["id"]
        tokens = preprocess_text(doc["text"])

        processed_docs[doc_id] = tokens
        doc_ids.append(doc_id)

        texts_for_tfidf.append(" ".join(tokens))

        # build inverted index
        term_counts = {}
        for t in tokens:
            term_counts[t] = term_counts.get(t, 0) + 1

        for term, tf in term_counts.items():
            inverted_index[term].append((doc_id, tf))

    # TF-IDF lexical vectors
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts_for_tfidf)

    # dense semantic index
    texts = [doc["text"] for doc in D]
    encoder = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = encoder.encode(texts)

    I = {
        "doc_ids": doc_ids,
        "inverted_index": inverted_index,
        "tfidf_vectorizer": vectorizer,
        "tfidf_matrix": tfidf_matrix,
        "encoder": encoder,
        "embeddings": embeddings
    }

    indexing_time = time.time() - start

    memory_size = (
        sys.getsizeof(I)
        + embeddings.nbytes
    )

    return I, indexing_time, memory_size

In [7]:
I, build_time, mem_bytes = indexing(docs, args={"use_dense": True})
print("Index built in", build_time, "seconds")
print("Approximate index size:", mem_bytes / (1024**2), "MB")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6458.47it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index built in 186.4932141304016 seconds
Approximate index size: 3.2595367431640625 MB


B) **Extractive summarization**

*B.1 Summarization baseline solution: results for a given document*

In [ ]:
#code, statistics and/or charts here


*B.2 IR models (TF-IDF, BM25, lightweight encoder LLMs)*

In [ ]:
#code, statistics and/or charts here

*B.4 Maximal Marginal Relevance*

In [ ]:
#code, statistics and/or charts here

C) **Abstractive summarization**

*C.1 Prompting decoder LLMs*

In [ ]:
#code, statistics and/or charts here

*C.2 Reciprocal rank funsion*

In [ ]:
#code, statistics and/or charts here

D) **Keyword extraction**

In [ ]:
#code, statistics and/or charts here

E) **Evaluation**

In [ ]:
#code, statistics and/or charts here

<H3>Part II: questions materials (optional)</H3>

**(1)** Keyword extraction using classic *vs* generative stances

In [ ]:
#code, statistics and/or charts here

**(2)** Summarization performance for the overall and category-conditional corpora.

In [ ]:
#code, statistics and/or charts here

**...** (additional questions with empirical results)

<H3>END</H3>